# Capstone add-on — Forward test on the full warehouse

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/susheel123-sketch/Flyrank-Internship-ml/blob/main/work/notebooks/w08_warehouse_forward_test.ipynb?flush_cache=true)

**Question:** does my final ranker (Logistic Regression on data from *before* the outcome window) still beat the rule and the base rate
on the full FlyRank warehouse — on **clients it never saw** and on a **later period than it was trained on**?

| | Features from | Outcome ("declining") |
|---|---|---|
| **Train period** | December 2025 | February impressions < 80% of January |
| **Test period** | January 2026 | March impressions < 80% of February |

- Features always come from the month **before** the outcome window starts → no overlap (the Week-6 lesson).
- Test period is **later** than train period, and test clients are **never** in training (GroupKFold by client) → a forward, unseen-client test.
- Only rows where Search Console data exists (`gsc_data_available IS TRUE`). The final month (June 2026) is not touched.
- Same label rule (>20% drop) and same metric (precision@K vs base rate) as the paper.

**Runs in Colab with a Colab Secret named `HF_TOKEN`.** Expect 5–15 minutes (four months are aggregated inside DuckDB). Seed 42.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os, json, getpass, warnings
import numpy as np, pandas as pd, duckdb
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
SEED = 42

con = duckdb.connect()
LOCAL = os.environ.get("WAREHOUSE_LOCAL")          # only used for an offline dry run; Colab uses Hugging Face
if LOCAL:
    REL = LOCAL
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        try:
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN")
        except Exception:
            pass
    HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    REL = "hf://datasets/FlyRank/internship-warehouse"
print("source:", "local dry-run copy" if LOCAL else "Hugging Face warehouse")

## 1. Aggregate four months per page, inside DuckDB

One row per page per month: impressions, clicks, days visible, impression-weighted position. Only rows with real Search Console data.
Results are cached to disk, so a re-run skips the download.

In [ ]:
MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03"]
CACHE = "/tmp/w08_cache"; os.makedirs(CACHE, exist_ok=True)

def month_agg(m):
    path = f"{CACHE}/{m}.parquet"
    if os.path.exists(path):
        return pd.read_parquet(path)
    src = f"read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')"
    df = con.sql(f'''
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                                   AS impr,
               SUM(gsc_clicks)                                        AS clicks,
               COUNT(*) FILTER (WHERE gsc_impressions > 0)            AS days_visible,
               SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS position
        FROM {src}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2''').df()
    df.to_parquet(path); return df

M = {}
for m in MONTHS:
    M[m] = month_agg(m)
    print(f"{m}: {len(M[m]):,} pages with Search Console data, {M[m]['impr'].sum():,.0f} impressions")
for m, d in M.items():
    assert not d.duplicated(["client_hash_id", "content_hash_id"]).any(), f"grain broken in {m}"
print("grain check: one row per client x page per month (asserted)")

dim_content = con.sql(f"SELECT * FROM read_parquet('{REL}/dim_content.parquet')").df()
dim_clients = con.sql(f"SELECT * FROM read_parquet('{REL}/dim_clients.parquet')").df()
print(f"dim_content {len(dim_content):,} rows | dim_clients {len(dim_clients):,} rows")

## 2. Build the two periods

Population per period: pages with ≥1 impression in the outcome's **base month** (otherwise "decline" is undefined), from clients whose
Search Console history starts on or before 1 Dec 2025, created before the feature month began.
Features use **only** the feature month plus page metadata.

In [ ]:
dim_clients["gsc_data_start"] = pd.to_datetime(dim_clients["gsc_data_start"], errors="coerce")
ok_clients = set(dim_clients.loc[dim_clients["gsc_data_start"] <= "2025-12-01", "client_hash_id"])
print(f"clients with GSC history from Dec 2025 or earlier: {len(ok_clients)} of {dim_clients['client_hash_id'].nunique()}")

meta_cols = [c for c in ["content_hash_id", "word_count", "search_volume", "content_type", "content_created_date"] if c in dim_content.columns]
meta = dim_content[meta_cols].drop_duplicates("content_hash_id")
meta["content_created_date"] = pd.to_datetime(meta["content_created_date"], errors="coerce")

def build_period(feat_m, base_m, out_m):
    key = ["client_hash_id", "content_hash_id"]
    base = M[base_m][key + ["impr"]].rename(columns={"impr": "base_impr"})
    base = base[(base["base_impr"] > 0) & base["client_hash_id"].isin(ok_clients)]
    feat = M[feat_m].rename(columns={"impr": "f_impr", "clicks": "f_clicks", "days_visible": "f_days", "position": "f_pos"})
    out = M[out_m][key + ["impr", "clicks"]].rename(columns={"impr": "out_impr", "clicks": "out_clicks"})
    d = base.merge(feat, on=key, how="left").merge(out, on=key, how="left").merge(meta, on="content_hash_id", how="left")
    for c in ["f_impr", "f_clicks", "f_days", "out_impr", "out_clicks"]:
        d[c] = d[c].fillna(0)
    start = pd.Timestamp(feat_m + "-01")
    d = d[d["content_created_date"].isna() | (d["content_created_date"] < start)].copy()
    d["age_at_cutoff"] = (start - d["content_created_date"]).dt.days
    d["y"] = (d["out_impr"] < 0.8 * d["base_impr"]).astype(int)
    return d.reset_index(drop=True)

A = build_period("2025-12", "2026-01", "2026-02")   # train period
B = build_period("2026-01", "2026-02", "2026-03")   # test period (later)
for name, d in [("TRAIN period (Dec -> Feb vs Jan)", A), ("TEST period (Jan -> Mar vs Feb)", B)]:
    print(f"{name}: {len(d):,} pages, {d['client_hash_id'].nunique()} clients, base rate {d['y'].mean():.3f}")

In [ ]:
def features(d):
    X = pd.DataFrame(index=d.index)
    X["impr_feat"] = np.log1p(d["f_impr"]); X["clicks_feat"] = np.log1p(d["f_clicks"])
    X["ctr_feat"] = np.where(d["f_impr"] > 0, d["f_clicks"] / d["f_impr"].replace(0, 1) * 100, 0)
    X["days_visible_feat"] = d["f_days"]; X["visible_feat"] = (d["f_impr"] > 0).astype(int)
    X["position_feat"] = d["f_pos"].fillna(0); X["has_position_feat"] = d["f_pos"].notna().astype(int)
    X["age_at_cutoff"] = d["age_at_cutoff"].fillna(d["age_at_cutoff"].median()); X["has_age"] = d["age_at_cutoff"].notna().astype(int)
    for c in ["word_count", "search_volume"]:
        if c in d:
            X[c] = np.log1p(pd.to_numeric(d[c], errors="coerce")).fillna(0); X["has_" + c] = d[c].notna().astype(int)
    if "content_type" in d:
        X = X.join(pd.get_dummies(d["content_type"].fillna("unknown"), prefix="ctype", dtype=int))
    return X

XA, XB = features(A), features(B)
XA, XB = XA.align(XB, join="outer", axis=1, fill_value=0)
FORBIDDEN = {"base_impr", "out_impr", "out_clicks", "y"}
assert not FORBIDDEN & set(XA.columns), "outcome-window column in features"
print(len(XA.columns), "features, all from the feature month or page metadata:", list(XA.columns))

## 3. Forward test, grouped by client

For each of 5 client folds: fit on the **train period** using only the training clients, then score the **test period** for the held-out
clients. The rule's three position-band CTR medians are also learned from training clients in the train period only.

In [ ]:
def rule_score(train_d, test_d):
    band = lambda d: pd.cut(d["f_pos"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
    ctr = lambda d: np.where(d["f_impr"] > 0, d["f_clicks"] / d["f_impr"].replace(0, 1) * 100, 0)
    bt = band(train_d); okt = (train_d["f_impr"] >= 167) & bt.notna()      # 167/month ~ 500 per 90 days
    med = pd.Series(ctr(train_d)[okt.values], index=train_d.index[okt]).groupby(bt[okt], observed=True).median()
    be = band(test_d); e = be.map(med).astype(float).values; c = ctr(test_d)
    flag = (test_d["f_impr"].values >= 167) & be.notna().values & (c < 0.5 * e)
    gap = np.clip(1 - c / np.where(e > 0, e, np.nan), 0, 1)
    return np.where(flag, np.nan_to_num(gap) * np.log1p(test_d["f_impr"].values), 0) + np.log1p(test_d["f_impr"].values) * 1e-6

MODELS = {"Logistic Regression (final)": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
          "Gradient Boosting": lambda: HistGradientBoostingClassifier(random_state=SEED)}
clientsB = B["client_hash_id"].values
scores = {k: np.full(len(B), np.nan) for k in ["Rule (CTR below position norm)"] + list(MODELS)}
for tr_idx, te_idx in GroupKFold(5).split(XB, B["y"], clientsB):
    train_clients = set(clientsB[tr_idx]); test_clients = set(clientsB[te_idx])
    assert not train_clients & test_clients
    a_mask = A["client_hash_id"].isin(train_clients).values
    scores["Rule (CTR below position norm)"][te_idx] = rule_score(A[a_mask], B.iloc[te_idx])
    for name, make in MODELS.items():
        m = make().fit(XA[a_mask], A["y"].values[a_mask])
        scores[name][te_idx] = m.predict_proba(XB.iloc[te_idx])[:, 1]
yB = B["y"].values
assert not any(np.isnan(s).any() for s in scores.values())
print("every test-period page scored by a model trained on earlier data from other clients (asserted)")

In [ ]:
def p_at_k(s, k, mask=None):
    yy, ss = (yB, s) if mask is None else (yB[mask], s[mask])
    return float(yy[np.argsort(-ss, kind="stable")[:k]].mean())

def capped_top(s, k=50, cap=10):
    order, taken, out = np.argsort(-s, kind="stable"), {}, []
    for i in order:
        c = clientsB[i]
        if taken.get(c, 0) < cap:
            taken[c] = taken.get(c, 0) + 1; out.append(i)
        if len(out) == k: break
    return np.array(out)

KS = [20, 50, 100, 500, 1000]
rows = [{"ranker": "Random order (base rate)", **{f"P@{k}": round(yB.mean(), 3) for k in KS}, "AUC": 0.5, "capped P@50": round(yB.mean(), 3)}]
for name, s in scores.items():
    rows.append({"ranker": name, **{f"P@{k}": round(p_at_k(s, k), 3) for k in KS},
                 "AUC": round(roc_auc_score(yB, s), 3), "capped P@50": round(yB[capped_top(s)].mean(), 3)})
res = pd.DataFrame(rows).set_index("ranker"); display(res)

big = (B["base_impr"] >= 20).values      # Week-6 label-noise fix: ignore 'declines' from tiny pages
rob = pd.DataFrame({name: [round(p_at_k(s, 50, big), 3), round(roc_auc_score(yB[big], s[big]), 3)] for name, s in scores.items()},
                   index=["P@50 (base >= 20 impressions)", "AUC (base >= 20)"]).T
rob["base rate (base >= 20)"] = round(yB[big].mean(), 3); display(rob)

final = scores["Logistic Regression (final)"]
top50 = capped_top(final)
print(f"Capped top 50 of the final model: {yB[top50].sum()} declined | {B['client_hash_id'].iloc[top50].nunique()} clients | "
      f"feature-month clicks covered: {int(B['f_clicks'].iloc[top50].sum())} | zero-click share: {(B['f_clicks'].iloc[top50] == 0).mean():.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for name, st in zip(scores, ["-s", "-o", "-^"]):
    ax.plot(KS, [p_at_k(scores[name], k) for k in KS], st, label=name, ms=4)
ax.axhline(yB.mean(), ls="--", c="grey", lw=1, label="random order (base rate)")
ax.set_xscale("log"); ax.set_xlabel("pages reviewed (top K, log scale)"); ax.set_ylabel("share that declined")
ax.set_title("Warehouse forward test: later period, unseen clients", fontsize=10); ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); plt.show()

receipt = {"assignment": "capstone warehouse forward test", "seed": SEED,
           "train_period": "features 2025-12, outcome 2026-02 vs 2026-01", "test_period": "features 2026-01, outcome 2026-03 vs 2026-02",
           "split": "GroupKFold(5) by client_hash_id; train rows from train period only",
           "n_train_pages": int(len(A)), "n_test_pages": int(len(B)), "n_test_clients": int(B["client_hash_id"].nunique()),
           "base_rate_test": round(float(yB.mean()), 4), "results": res.reset_index().to_dict(orient="records"),
           "robustness_base_ge_20": rob.reset_index().rename(columns={"index": "ranker"}).to_dict(orient="records"),
           "capped_top50_final": {"declined": int(yB[top50].sum()), "clients": int(B["client_hash_id"].iloc[top50].nunique()),
                                  "feature_month_clicks": int(B["f_clicks"].iloc[top50].sum())}}
print("RECEIPT_JSON_START"); print(json.dumps(receipt, default=float)); print("RECEIPT_JSON_END")

## 4. Reading the result

*(Filled in after the run — the numbers above decide what this section says.)*